In [ ]:
# VectorDB에 이미지 저장 후 검색 - 이미지로 이미지 검색, 텍스트로 이미지 검색
# !pip install huggingface_hub

from huggingface_hub import list_models
models = list_models(search="clip", limit=20)
for m in models:
    print(m.modelId)

In [ ]:
# clip 모델로 이미지 임베딩
!pip install chromadb sentence-transformers torch pillow transformers
!pip install koreanize-matplotlib

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import koreanize_matplotlib
from chromadb import PersistentClient
from transformers import CLIPProcessor, CLIPModel
from google.colab import files
from numpy.linalg import norm
from PIL import Image

# image upload
uploaded = files.upload()


In [ ]:
# CLIP 모델 준비
model_name = "openai/clip-vit-base-patch32"   # 허깅페이스에 등록된 CLIP 기본 모델
processor = CLIPProcessor.from_pretrained(model_name)   # 데이터를 CLIP 입력형식으로 전처리
model = CLIPModel.from_pretrained(model_name)  # 데이터를 밀집벡터로 변환

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)  # 모델을 선택한 장치로 이동함
model.eval()  # 학습용 모델이 아니라 추론용으로 사용

print('모델이름:', model_name)
print('사용장치:', device)
print('모델타입:', type(model))


In [ ]:
# 이미지 -> CLIP 벡터로 변환
def image_to_vector(img_path):
    image = Image.open(img_path).convert('RGB')

    inputs = processor(   # 이미지를 CLIP 모델 입력형식으로 변환
        images = image,   # 전처리 대상 이미지 지정
        return_tensors="pt"   # 결과를 PyTorch 텐서형식으로 반환
    )
    pixel_values = inputs['pixels_values'].to(device)  # 이미지 텐서를 모델과 같은 장치로 이동

    with torch.no_grad():  # 역전파를 끄고(기울기 계산X) 추론만 하겠다는 의미
        vision_outputs = model.vision_model(  # CLIP 이미지 인코더로 이미지 특징 추출
            pixel_values = pixel_values
        )
        # vision_model : 이미지를 보고 정보를 만듦
        # vision_outputs.last_hidden_state : 이미지를 패치 단위로 나눔
        # vision_outputs.pooler_output : 이미지 전체를 대표하는 하나의 벡터
        pooled_putput = vision_outputs.pooler_output  # 이미지 전체를 대표하는 벡터 꺼내기
        image_features = model.visual_projection(  # 이미지벡터를 CLIP 공통 임베딩 공간으로 변환
            pooled_putput
        )

    image_features = image_features / image_features.norm(  # 벡터 크기를 1로 정규화
        p=2,   # L2 norm 사용
        dim = -1,
        keepdim = True
    )

    vec = image_features.squeeze(0).cpu().numpy()  # 배치 차원을 제거하고 cpu로 이동 후 numpy배열로 반환
    print(f'\n{img_path} -> 이미지 벡터 크기:{vec.shape}')
    print(f'{img_path} -> 벡터 앞 10개: {vec[:10]}')
    return vec.tolist()   # ChromaDB에 저장 가능한 벡터를 반환

# 텍스트 -> CLIP 벡터로 변환
def text_to_vector(text):
    pass

In [ ]:
# 이미지 -> CLIP 벡터로 변환
def image_to_vector(img_path):
    image = Image.open(img_path).convert('RGB')

    inputs = processor(   # 이미지를 CLIP 모델 입력형식으로 변환
        images = image,   # 전처리 대상 이미지 지정
        return_tensors="pt"   # 결과를 PyTorch 텐서형식으로 반환
    )
    pixel_values = inputs['pixel_values'].to(device)  # 이미지 텐서를 모델과 같은 장치로 이동

    with torch.no_grad():  # 역전파를 끄고(기울기 계산X) 추론만 하겠다는 의미
        vision_outputs = model.vision_model(  # CLIP 이미지 인코더로 이미지 특징 추출
            pixel_values = pixel_values
        )
        # vision_model : 이미지를 보고 정보를 만듦
        # vision_outputs.last_hidden_state : 이미지를 패치 단위로 나눔
        # vision_outputs.pooler_output : 이미지 전체를 대표하는 하나의 벡터
        pooled_putput = vision_outputs.pooler_output  # 이미지 전체를 대표하는 벡터 꺼내기
        image_features = model.visual_projection(  # 이미지벡터를 CLIP 공통 임베딩 공간으로 변환
            pooled_putput
        )

    image_features = image_features / image_features.norm(  # 벡터 크기를 1로 정규화
        p=2,   # L2 norm 사용
        dim = -1,
        keepdim = True
    )

    vec = image_features.squeeze(0).cpu().numpy()  # 배치 차원을 제거하고 cpu로 이동 후 numpy배열로 반환
    print(f'\n{img_path} -> 이미지 벡터 크기:{vec.shape}')
    print(f'{img_path} -> 벡터 앞 10개: {vec[:10]}')
    return vec.tolist()   # ChromaDB에 저장 가능한 벡터를 반환

# 텍스트 -> CLIP 벡터로 변환
def text_to_vector(text):
    inputs = processor(   # 텍스트를 CLIP 모델 입력형식으로 변환
        text = text,   # 전처리 대상 텍스트 지정
        return_tensors="pt",  # 결과를 PyTorch 텐서형식으로 반환
        padding=True
    )
    input_ids = inputs['input_ids'].to(device)  # 텍스트 텐서를 모델과 같은 장치로 이동
    attention_mask = inputs['attention_mask'].to(device)  # 실제 토큰과 패딩을 구분하는 마스크

    with torch.no_grad():  # 역전파를 끄고(기울기 계산X) 추론만 하겠다는 의미
        text_outputs = model.text_model(  # CLIP 텍스트 인코더로 텍스트 특징 추출
            input_ids = input_ids,   # 텍스트 토큰 id 전달
            attention_mask = attention_mask
        )
        # text_model : 텍스트를 보고 정보를 만듦
        pooled_putput = text_outputs.pooler_output  # 문장 전체를 대표하는 벡터 꺼내기
        text_features = model.text_projection(  # 텍스트벡터를 CLIP 공통 임베딩 공간으로 변환
            pooled_putput
        )

    text_features = text_features / text_features.norm(  # 벡터 크기를 1로 정규화
        p=2,   # L2 norm 사용
        dim = -1,
        keepdim = True
    )

    vec = text_features.squeeze(0).cpu().numpy()  # 배치 차원을 제거하고 cpu로 이동 후 numpy배열로 반환
    print(f'\n검색어 {text} -> 텍스트 벡터 크기:{vec.shape}')
    print(f'검색어 {text} -> 벡터 앞 10개: {vec[:10]}')
    return vec.tolist()   # ChromaDB에 저장 가능한 벡터를 반환



In [ ]:
# chromaDb 초기화
client = PersistentClient(path="./chroma_clip")

try:
    client.delete_collection(name="images_clip")
except:
    pass

collection = client.get_or_create_collection(
    name = "images_clip",
    metadata={"hnsw:space":"cosine"}
)

# 업로드한 이미지 목록 준비
images_files = ['apple.jpeg','banana.jpeg','peach.jpeg']
ids = [f'img{i}' for i in range(len(images_files))]
print('ids : ', ids)

image_vectors = {}  # 계산된 이미지 저장용

# chroma에 저장
for img_id, img_path in zip(ids, images_files):
    if not os.path.exists(img_path):
        print(f"파일없음 {img_path}")
        continue

    vec = image_to_vector(img_path)   # 이미지를 Clip벡터로 변환
    image_vectors[img_path] = vec

    collection.upsert(  # 같은 id가 있으면 수정, 없으면 추가
        embeddings=[vec],
        documents=[img_path],
        ids=[img_id],
        metadatas=[{'filename':img_path}]
    )

record = collection.get(
    ids = ['img0'],
    include=['embeddings', 'documents', 'metadatas']
)

if len(record['ids']) > 0:
    print('\n벡터 DB에 저장된 내용 확인')
    print('id : ', record['ids'][0])
    print('문서 : ', record['documents'][0])
    print('정보 : ', record['metadatas'][0])
    print('벡터(앞 10개) : ', record['embeddings'][0][:10])
else:
    print('저장된 자료가 없어요')



In [ ]:
# 코사인 유사도 계산 함수 (방향유사도 = 코사인유사도 = 내적 / (A의길이 * B의 길이))
def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    return np.dot(vec1, vec2) / (norm(vec1) * norm(vec2))  # CLIP이 이 방법 사용

# 검색 결과 출력 함수
def print_search_results(title, query_vec, results):
    print('\n' + title)
    for rank, (doc, meta, dist, vec) in enumerate(
        zip(
            results['documents'][0],
            results['metadatas'][0],
            results['distances'][0],
            results['embeddings'][0]
         ),
        start=1
    ):
        score = cosine_similarity(query_vec, vec)
        print(f'\n{rank}위')
        print(f'  - 파일명:{meta['filename']}')
        print(f'  - 문서:{doc}')
        print(f'  - chroma 거리:{dist:.4f}')
        print(f'  - cosine 유사도:{1 - score:.4f}')


# 이미지로 유사 이미지 검색 -----------------------
query_image_path = 'berry.jpeg'   # 검색 기준으로 사용할 이미지

if not os.path.exists(query_image_path):
    print('검색 이미지가 없어요')
elif collection.count() == 0:
    print('DB에 검색할 이미지가 없어요')
else:
    query_vec = image_to_vector(query_image_path)
    if query_vec is None:
        query_vec = image_to_vector(query_image_path)

    image_results = collection.query(
        query_embeddings = [query_vec],
        n_results = min(3, collection.count()),
        include=['metadatas', 'distances', 'documents', 'embeddings']
    )

    print_search_results(
        title=f'유사 이미지 검색 결과:{query_image_path}',
        query_vec = query_vec,
        results=image_results
    )


In [ ]:
# 테스트로 유사 이미지 검색 -----------------------
query_text = 'show me a picture of red fruits'  # Clip은 영어 검색어가 더 안정적

if collection.count() == 0:
    print('검색 이미지 없어요')
else:
    text_vec = text_to_vector(query_text)

    text_results = collection.query(
        query_embeddings = [text_vec],
        n_results = min(3, collection.count()),
        include=['metadatas', 'distances', 'documents', 'embeddings']
    )

    print_search_results(
        title=f'텍스트에 대한 유사 이미지 검색 결과:{query_text}',
        query_vec = text_vec,
        results=text_results
    )



In [ ]:
# 검색 결과 이미지로 시각화
def show_results(query_title, results):
    count = len(results['documents'][0])
    print('count : ', count)

    fig, axes = plt.subplots(1, count, figsize=(5 * count, 5))
    if count == 1:
        axes = [axes]

    for i, (doc, meta, dist) in enumerate(
        zip(results['documents'][0], results['metadatas'][0],results['distances'][0])
    ):
        axes[i].imshow(Image.open(doc).convert("RGB"))
        axes[i].set_title(f"{i + 1}위\n파일명:{meta['filename']}\n거리:{dist:.4f}")
        axes[i].axis('off')
    plt.suptitle(query_title)
    plt.show()


# 이미지 검색 결과 시각화
if "image_results" in globals():
    show_results(f'이미지 검색 결과:{query_image_path}', image_results)

# 텍스트 검색 결과 시각화
if "text_results" in globals():
    show_results(f'텍스트 검색 결과:{query_text}', text_results)